# Multi-Regressor Pipeline (Train/Val/Test 80/10/10)
Notebook ini mencakup:
1. Load dataset hasil encoding
2. Split data: train 80%, validation 10%, test 10%
3. Standardisasi fitur dan target
4. Simpan split data, scaler, metadata scaler
5. Training MultiOutput Regressor + evaluasi

In [1]:
from pathlib import Path
from time import perf_counter
import json
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [2]:
# Konfigurasi path
step_start = perf_counter()

DATA_PATH = Path('outlier_treatment_outputs/java-ash-hysplit_handled.csv')
ARTIFACT_DIR = Path('artifacts_multi_regressor_outlier')
SPLIT_DIR = ARTIFACT_DIR / 'splits'
SCALER_DIR = ARTIFACT_DIR / 'scalers'
MODEL_DIR = ARTIFACT_DIR / 'model'
METRIC_DIR = ARTIFACT_DIR / 'metrics'

for d in [ARTIFACT_DIR, SPLIT_DIR, SCALER_DIR, MODEL_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
process_times = {}

process_times['setup_paths'] = perf_counter() - step_start
print(f"Waktu setup_paths: {process_times['setup_paths']:.6f} detik")

Waktu setup_paths: 0.001889 detik


In [3]:
# Load data
step_start = perf_counter()

df = pd.read_csv(DATA_PATH)

process_times['load_data'] = perf_counter() - step_start
print(f"Waktu load_data: {process_times['load_data']:.6f} detik")
print('Shape data:', df.shape)
df.head()

Waktu load_data: 0.009230 detik
Shape data: (1707, 20)


,id,timestamp,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km,volcano_filter_Bromo,volcano_filter_Merapi,volcano_filter_Raung,volcano_filter_Semeru,volcano_filter_Slamet
0,1,6,3,-7.54194,110.44194,2968,1000.0,4.7,270.0,50.0,66.5,32.511505,614.390650,303.257335,18.304149,0,1,0,0,0
1,2,1,1,-7.54194,110.44194,2968,1000.0,8.0,135.0,51.0,168.0,27.761756,460.792988,37.901873,15.892819,0,1,0,0,0
2,3,3,1,-7.54194,110.44194,2968,1200.0,7.1,90.0,65.0,133.0,44.625714,614.214150,265.635506,23.934437,0,1,0,0,0
3,4,3,1,-7.54194,110.44194,2968,1250.0,3.2,135.0,70.0,160.0,22.998759,184.285530,288.108280,11.770659,0,1,0,0,0
4,5,3,1,-7.54194,110.44194,2968,1250.0,8.3,315.0,70.0,145.0,39.159595,307.107075,270.462663,20.809694,0,1,0,0,0


In [5]:
# Pilih target multi-output (sesuaikan jika ingin target lain)
step_start = perf_counter()

target_cols = ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']

missing_targets = [c for c in target_cols if c not in df.columns]
if missing_targets:
    raise ValueError(f'Target tidak ditemukan di dataset: {missing_targets}')

drop_cols = target_cols + ['id']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[target_cols].copy()

process_times['prepare_features_targets'] = perf_counter() - step_start
print(f"Waktu prepare_features_targets: {process_times['prepare_features_targets']:.6f} detik")
print('Kolom yang di-drop dari fitur:', ['id'])
print('Jumlah fitur:', len(feature_cols))
print('Fitur:', feature_cols)
print('Target:', target_cols)

Waktu prepare_features_targets: 0.002595 detik
Kolom yang di-drop dari fitur: ['id']
Jumlah fitur: 15
Fitur: ['timestamp', 'alert_level', 'latitude', 'longitude', 'elevation', 'tinggi_letusan_m', 'kec_angin_km_jam', 'arah_angin_deg', 'amplitudo', 'duration', 'volcano_filter_Bromo', 'volcano_filter_Merapi', 'volcano_filter_Raung', 'volcano_filter_Semeru', 'volcano_filter_Slamet']
Target: ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']


In [6]:
# Split 80/10/10: pertama pisah train 80% dan temp 20%, lalu temp dibagi dua untuk val & test
step_start = perf_counter()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE
)

process_times['split_data'] = perf_counter() - step_start
print(f"Waktu split_data: {process_times['split_data']:.6f} detik")
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}, y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

Waktu split_data: 0.002607 detik
X_train: (1365, 15), y_train: (1365, 4)
X_val:   (171, 15), y_val:   (171, 4)
X_test:  (171, 15), y_test:  (171, 4)


In [7]:
# Standardisasi (fit hanya di data train)
step_start = perf_counter()

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = pd.DataFrame(x_scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(x_scaler.transform(X_val), columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(x_scaler.transform(X_test), columns=feature_cols, index=X_test.index)

y_train_scaled = pd.DataFrame(y_scaler.fit_transform(y_train), columns=target_cols, index=y_train.index)
y_val_scaled = pd.DataFrame(y_scaler.transform(y_val), columns=target_cols, index=y_val.index)
y_test_scaled = pd.DataFrame(y_scaler.transform(y_test), columns=target_cols, index=y_test.index)

process_times['standardize_data'] = perf_counter() - step_start
print(f"Waktu standardize_data: {process_times['standardize_data']:.6f} detik")
print('Standardisasi selesai.')

Waktu standardize_data: 0.013602 detik
Standardisasi selesai.


In [8]:
# Simpan data split mentah
step_start = perf_counter()

X_train.to_csv(SPLIT_DIR / 'X_train_raw.csv', index=False)
X_val.to_csv(SPLIT_DIR / 'X_val_raw.csv', index=False)
X_test.to_csv(SPLIT_DIR / 'X_test_raw.csv', index=False)
y_train.to_csv(SPLIT_DIR / 'y_train_raw.csv', index=False)
y_val.to_csv(SPLIT_DIR / 'y_val_raw.csv', index=False)
y_test.to_csv(SPLIT_DIR / 'y_test_raw.csv', index=False)

# Simpan data split hasil standardisasi
X_train_scaled.to_csv(SPLIT_DIR / 'X_train_scaled.csv', index=False)
X_val_scaled.to_csv(SPLIT_DIR / 'X_val_scaled.csv', index=False)
X_test_scaled.to_csv(SPLIT_DIR / 'X_test_scaled.csv', index=False)
y_train_scaled.to_csv(SPLIT_DIR / 'y_train_scaled.csv', index=False)
y_val_scaled.to_csv(SPLIT_DIR / 'y_val_scaled.csv', index=False)
y_test_scaled.to_csv(SPLIT_DIR / 'y_test_scaled.csv', index=False)

process_times['save_splits'] = perf_counter() - step_start
print(f"Waktu save_splits: {process_times['save_splits']:.6f} detik")
print('Semua split data berhasil disimpan di folder splits.')

Waktu save_splits: 0.130622 detik
Semua split data berhasil disimpan di folder splits.


In [9]:
# Simpan scaler (format PKL)
step_start = perf_counter()

with open(SCALER_DIR / 'x_scaler.pkl', 'wb') as f:
    pickle.dump(x_scaler, f)

with open(SCALER_DIR / 'y_scaler.pkl', 'wb') as f:
    pickle.dump(y_scaler, f)

# Simpan metadata scaler
x_scaler_metadata = {
    'feature_names': feature_cols,
    'mean': x_scaler.mean_.tolist(),
    'scale': x_scaler.scale_.tolist(),
    'var': x_scaler.var_.tolist(),
    'n_features_in': int(x_scaler.n_features_in_),
}

y_scaler_metadata = {
    'target_names': target_cols,
    'mean': y_scaler.mean_.tolist(),
    'scale': y_scaler.scale_.tolist(),
    'var': y_scaler.var_.tolist(),
    'n_features_in': int(y_scaler.n_features_in_),
}

with open(SCALER_DIR / 'x_scaler_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(x_scaler_metadata, f, indent=2)

with open(SCALER_DIR / 'y_scaler_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(y_scaler_metadata, f, indent=2)

process_times['save_scalers_and_metadata'] = perf_counter() - step_start
print(f"Waktu save_scalers_and_metadata: {process_times['save_scalers_and_metadata']:.6f} detik")
print('Scaler PKL dan metadata scaler berhasil disimpan.')

Waktu save_scalers_and_metadata: 0.003997 detik
Scaler PKL dan metadata scaler berhasil disimpan.


In [10]:
# Training multi-regressor XGBoost pada data yang sudah distandardisasi
step_start = perf_counter()

xgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
    'objective': 'reg:squarederror',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

base_model = XGBRegressor(**xgb_params)
model = MultiOutputRegressor(base_model)
model.fit(X_train_scaled, y_train_scaled)

with open(MODEL_DIR / 'multi_output_xgboost.pkl', 'wb') as f:
    pickle.dump(model, f)

with open(MODEL_DIR / 'xgboost_hyperparameters.json', 'w', encoding='utf-8') as f:
    json.dump(xgb_params, f, indent=2)

process_times['train_model'] = perf_counter() - step_start
print(f"Waktu train_model: {process_times['train_model']:.6f} detik")
print('Model XGBoost berhasil dilatih dan disimpan.')
print('Hyperparameter XGBoost:', xgb_params)

Waktu train_model: 0.807919 detik
Model XGBoost berhasil dilatih dan disimpan.
Hyperparameter XGBoost: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}


In [11]:
# Evaluasi di validation dan test (dikembalikan ke skala asli target)
step_start = perf_counter()

def evaluate_split(split_name, X_split_scaled, y_split_raw):
    y_pred_scaled = model.predict(X_split_scaled)
    y_pred_raw = y_scaler.inverse_transform(y_pred_scaled)

    rows = []
    for i, target in enumerate(target_cols):
        y_true = y_split_raw[target].values
        y_pred = y_pred_raw[:, i]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)

        rows.append({
            'split': split_name,
            'target': target,
            'MAE': float(mae),
            'RMSE': float(rmse),
            'R2': float(r2),
        })

    return pd.DataFrame(rows)

val_metrics = evaluate_split('val', X_val_scaled, y_val)
test_metrics = evaluate_split('test', X_test_scaled, y_test)
metrics_df = pd.concat([val_metrics, test_metrics], ignore_index=True)

metrics_df.to_csv(METRIC_DIR / 'metrics_per_target.csv', index=False)

summary = metrics_df.groupby('split')[['MAE', 'RMSE', 'R2']].mean().reset_index()
summary.to_csv(METRIC_DIR / 'metrics_summary.csv', index=False)

process_times['evaluate_and_save_metrics'] = perf_counter() - step_start
print(f"Waktu evaluate_and_save_metrics: {process_times['evaluate_and_save_metrics']:.6f} detik")
print('Metrik tersimpan di folder metrics.')
display(metrics_df)
display(summary)

Waktu evaluate_and_save_metrics: 0.042471 detik
Metrik tersimpan di folder metrics.


,split,target,MAE,RMSE,R2
0,val,jarak_km,6.415702,10.576203,0.750913
1,val,luas_km2,170.862933,255.607253,0.526799
2,val,sudut_deg,42.467268,67.543055,0.630496
3,val,radius_km,3.351424,6.019528,0.724751
4,test,jarak_km,6.893095,13.742507,0.762393
5,test,luas_km2,236.367838,427.905705,0.169750
6,test,sudut_deg,48.314624,76.598854,0.474897
7,test,radius_km,3.777294,7.554127,0.769502


,split,MAE,RMSE,R2
0,test,73.838213,131.450298,0.544136
1,val,55.774332,84.936510,0.658240


In [12]:
# Ringkasan file artefak yang tersimpan
step_start = perf_counter()

saved_files = sorted([str(p) for p in ARTIFACT_DIR.rglob('*') if p.is_file()])
print('Daftar file artefak:')
for f in saved_files:
    print('-', f)

process_times['list_artifacts'] = perf_counter() - step_start
print(f"Waktu list_artifacts: {process_times['list_artifacts']:.6f} detik")

# Simpan dan tampilkan rekap waktu proses
timing_df = pd.DataFrame(
    [{'process': k, 'duration_seconds': v} for k, v in process_times.items()]
 ).sort_values('duration_seconds', ascending=False).reset_index(drop=True)

timing_df.to_csv(METRIC_DIR / 'process_times.csv', index=False)
print("Rekap waktu proses disimpan ke:", METRIC_DIR / 'process_times.csv')
display(timing_df)

Daftar file artefak:
- artifacts_multi_regressor_outlier\metrics\metrics_per_target.csv
- artifacts_multi_regressor_outlier\metrics\metrics_summary.csv
- artifacts_multi_regressor_outlier\model\multi_output_xgboost.pkl
- artifacts_multi_regressor_outlier\model\xgboost_hyperparameters.json
- artifacts_multi_regressor_outlier\scalers\x_scaler.pkl
- artifacts_multi_regressor_outlier\scalers\x_scaler_metadata.json
- artifacts_multi_regressor_outlier\scalers\y_scaler.pkl
- artifacts_multi_regressor_outlier\scalers\y_scaler_metadata.json
- artifacts_multi_regressor_outlier\splits\X_test_raw.csv
- artifacts_multi_regressor_outlier\splits\X_test_scaled.csv
- artifacts_multi_regressor_outlier\splits\X_train_raw.csv
- artifacts_multi_regressor_outlier\splits\X_train_scaled.csv
- artifacts_multi_regressor_outlier\splits\X_val_raw.csv
- artifacts_multi_regressor_outlier\splits\X_val_scaled.csv
- artifacts_multi_regressor_outlier\splits\y_test_raw.csv
- artifacts_multi_regressor_outlier\splits\y_te

,process,duration_seconds
0,train_model,0.807919
1,save_splits,0.130622
2,evaluate_and_save_metrics,0.042471
3,standardize_data,0.013602
4,load_data,0.009230
5,save_scalers_and_metadata,0.003997
6,list_artifacts,0.002789
7,split_data,0.002607
8,prepare_features_targets,0.002595
9,setup_paths,0.001889
